In [ ]:
import tensorflow as tf
from tensorflow.keras import models, Model, layers
import numpy as np
import pandas as pd
import os
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_sample_weight
import itertools

# --- CONFIGURATION ---
BASE_PATH = "/content/drive/MyDrive/1 Skripsi/27jan/"

# Input: The 128-dim Learned Alpha Vector (from Pre-training)
VIEW1_PAYLOAD_PATH = os.path.join(BASE_PATH, "PRETRAIN_X_view1.npy")
RAW_LABELS_PATH = os.path.join(BASE_PATH, "cnn_payload_labels.csv")
MERGED_CSV_PATH = os.path.join(BASE_PATH, "merged_components_consistent.csv")

# Weights
WEIGHTS_PATH = os.path.join(BASE_PATH, "FULL_HMVCL_Encoder.weights.h5")

LABEL_PERCENTAGE = 0.20
LATENT_DIM = 128

# --- 1. Architecture ---
def get_cnn_encoder(input_shape):
    inputs = layers.Input(shape=input_shape)
    x = layers.Reshape((input_shape[0], 1))(inputs)

    # Architecture (Script B / MaxPool 2)
    x = layers.Conv1D(32, 7, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Conv1D(64, 5, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Conv1D(128, 3, activation='relu', padding='same')(x)
    x = layers.MaxPooling1D(2)(x)

    x = layers.Flatten()(x)
    h = layers.Dense(LATENT_DIM, activation='relu', name="representation")(x)
    z = layers.Dense(64, activation='relu')(h)
    return Model(inputs, [h, z])

# --- 2. Load & Align ---
def load_features():
    print("--- Preparing Data ---")
    if not os.path.exists(VIEW1_PAYLOAD_PATH): return None, None

    # Load Payload
    X_view1 = np.load(VIEW1_PAYLOAD_PATH).astype('float32')

    # Load & Clamp Labels
    df_labels = pd.read_csv(RAW_LABELS_PATH)
    if len(df_labels) > len(X_view1):
        df_labels = df_labels.iloc[:len(X_view1)]

    filename_map = {name: i for i, name in enumerate(df_labels['filename'])}

    # Load Stats
    df_merged = pd.read_csv(MERGED_CSV_PATH).fillna(0)

    valid_payloads = []
    aligned_indices = []

    for idx, row in df_merged.iterrows():
        fname = row['filename']
        if fname in filename_map:
            valid_payloads.append(X_view1[filename_map[fname]])
            aligned_indices.append(idx)

    X_payload = np.array(valid_payloads)
    df_final = df_merged.iloc[aligned_indices].reset_index(drop=True)

    return df_final, X_payload

# --- 3. Run Full Permutations ---
def run_full_ablation(df, X_payload, target_label):
    print(f"\n=== Full Permutation Ablation Study (15 Experiments) ===")

    # 1. Extract Alpha
    print("Extracting Alpha Features...")
    cnn = get_cnn_encoder((X_payload.shape[1],))
    if os.path.exists(WEIGHTS_PATH):
        try:
            cnn.load_weights(WEIGHTS_PATH)
        except:
            print("Warning: Random weights.")

    extractor = Model(inputs=cnn.input, outputs=cnn.outputs[0])
    X_alpha = extractor.predict(X_payload, batch_size=128, verbose=0)

    # 2. Define Feature Groups
    exclude = ['alpha_', 'application', 'category', 'binary_type', 'filename', 'temp_app', 'label', 'Binary', 'Category', 'App']

    feat_beta = [c for c in df.columns if c.startswith('beta_') and not any(x in c for x in exclude)]
    feat_gamma = [c for c in df.columns if c.startswith('gamma_') and not any(x in c for x in exclude)]
    feat_fft = [c for c in df.columns if c.startswith('fft_') and not any(x in c for x in exclude)]

    # Prepare Normalized Data Blocks
    # We pre-calculate these so we don't re-normalize 15 times
    data_blocks = {
        'Alpha': X_alpha,
        'Beta': StandardScaler().fit_transform(df[feat_beta].values.astype('float32')),
        'Gamma': StandardScaler().fit_transform(df[feat_gamma].values.astype('float32')),
        'FFT': StandardScaler().fit_transform(df[feat_fft].values.astype('float32'))
    }

    results = []

    # 3. Generate All Combinations (1 to 4 items)
    component_names = ['Alpha', 'Beta', 'Gamma', 'FFT']

    for r in range(1, 5): # Length 1, 2, 3, 4
        for combination in itertools.combinations(component_names, r):
            comb_name = "+".join(combination)

            # Fuse selected blocks
            selected_blocks = [data_blocks[name] for name in combination]
            X_final = np.concatenate(selected_blocks, axis=1)

            # Train
            X_train, X_test, y_train, y_test = train_test_split(
                X_final, target_label, train_size=LABEL_PERCENTAGE,
                random_state=42, stratify=target_label
            )

            weights = compute_sample_weight('balanced', y_train)

            # Fast XGBoost settings for ablation
            clf = xgb.XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.05, n_jobs=-1)
            clf.fit(X_train, y_train, sample_weight=weights)

            y_pred = clf.predict(X_test)
            score = f1_score(y_test, y_pred, average='weighted')

            print(f"   [{comb_name}]: {score:.4f}")
            results.append((comb_name, score))

    # 4. Print Sorted Summary
    print("\n=== Final Ranking ===")
    results.sort(key=lambda x: x[1], reverse=True)
    for name, score in results:
        print(f"{score:.4f} | {name}")

# --- 4. Main ---
def main():
    df, X_pay = load_features()
    if df is None: return

    # Label Logic
    app_col = next((c for c in df.columns if c.endswith('application')), None)
    final_apps = []
    for idx, row in df.iterrows():
        fname = str(row['filename']).lower()
        prefix = "VPN" if "vpn" in fname else "NonVPN"
        final_apps.append(f"{prefix}_{row[app_col]}")
    df['temp_app'] = final_apps

    target_apps = ['VPN_Skype', 'VPN_BitTorrent', 'VPN_Hangout', 'VPN_Facebook', 'VPN_YouTube', 'VPN_Email']
    mask = df['temp_app'].isin(target_apps)

    le = LabelEncoder()
    y = le.fit_transform(df.loc[mask, 'temp_app'])

    # Run on Top Apps (Hardest Task)
    run_full_ablation(df[mask], X_pay[mask], y)

if __name__ == "__main__":
    main()

--- Preparing Data ---

=== Full Permutation Ablation Study (15 Experiments) ===
Extracting Alpha Features...
   [Alpha]: 0.9029
   [Beta]: 0.8969
   [Gamma]: 0.8883
   [FFT]: 0.7889
   [Alpha+Beta]: 0.9267
   [Alpha+Gamma]: 0.9139
   [Alpha+FFT]: 0.9174
   [Beta+Gamma]: 0.8958
   [Beta+FFT]: 0.8931
   [Gamma+FFT]: 0.8851
   [Alpha+Beta+Gamma]: 0.9261
   [Alpha+Beta+FFT]: 0.9298
   [Alpha+Gamma+FFT]: 0.9186
   [Beta+Gamma+FFT]: 0.9004
   [Alpha+Beta+Gamma+FFT]: 0.9324

=== Final Ranking ===
0.9324 | Alpha+Beta+Gamma+FFT
0.9298 | Alpha+Beta+FFT
0.9267 | Alpha+Beta
0.9261 | Alpha+Beta+Gamma
0.9186 | Alpha+Gamma+FFT
0.9174 | Alpha+FFT
0.9139 | Alpha+Gamma
0.9029 | Alpha
0.9004 | Beta+Gamma+FFT
0.8969 | Beta
0.8958 | Beta+Gamma
0.8931 | Beta+FFT
0.8883 | Gamma
0.8851 | Gamma+FFT
0.7889 | FFT
